<a href="https://colab.research.google.com/github/tewei0328/teach-programming/blob/main/twtalk4_gem_python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import yfinance as yf
import pandas as pd
import numpy as np
from tqdm import tqdm
from datetime import datetime

# --- 1. 配置區：台股 ETF 觀察清單 ---
WATCHLIST = {
    "0050.TW": "元大台灣50",
    "0051.TW": "元大中型100",
    "0053.TW": "元大電子",
    "0056.TW": "元大高股息",
    "006203.TW": "元大MSCI台灣",
    "006204.TW": "永豐臺灣加權",
    "00713.TW": "元大台灣高息低波",
    "00728.TW": "第一金工業30",
    "00733.TW": "富邦臺灣中小",
    "00878.TW": "國泰永續高股息",
    "00881.TW": "國泰台灣5G+",
    "00891.TW": "中信關鍵半導體",
    "00894.TW": "中信小資高價30",
    "00900.TW": "富邦特選高股息30",
    "00901.TW": "永豐智能車供應鏈",
    "00904.TW": "新光臺灣半導體30",
    "00905.TW": "FT臺灣Smart高股息",
    "00919.TW": "群益台灣精選高息",
    "00929.TW": "復華台灣科技優息",
    "00935.TW": "野村臺灣創新細分",
    "00947.TW": "台新臺灣IC設計",
    "00981A.TW": "中信成長高股息",
    "00936.TW": "台新永續高息中小型",
    "00939.TW": "統一台灣高息動能",
    "00940.TW": "元大台灣價值高息",
    "00941.TW": "中信上游半導體",
    "00946.TW": "群益科技高息成長",
    "00992A.TW": "中信臺灣智慧半導體",
    "00991A.TW": "中信高股息",
    "00922.TW": "國泰智能電動車",
    "00896.TW": "中信綠能及電動車",
    "0052.TW": "富邦科技",
    "00923.TW": "群益半導體收益",
    "00735.TW": "國泰中國A50",
    "00663L.TW": "國泰臺灣加權正2",
    "00830.TW": "富邦台灣ETF",
    "00670L.TW": "富邦臺灣加權正2",
    "00917.TW": "中信ESG永續高股息",
    "00927.TW": "群益台灣科技高息成長",
    "00902.TW": "中信臺灣智慧高股息",
    "00690.TW": "富邦印度ETF"
}

def analyze_logic(ticker, df):
    """核心量化引擎：判定型態、籌碼、評分與建議"""
    last = df.iloc[-1]
    prev = df.iloc[-2]

    # A. 計算 20日主力成本 (VWMA)
    # 公式：(Close * Volume).rolling(20).sum() / Volume.rolling(20).sum()
    df['PV'] = df['Close'] * df['Volume']
    curr_vwma = df['PV'].rolling(20).sum() / df['Volume'].rolling(20).sum()
    curr_vwma_val = curr_vwma.iloc[-1]

    # B. 技術指標計算
    bias_pct = ((last['Close'] / curr_vwma_val) - 1) * 100
    day1_ret = (last['Close'] / prev['Close'] - 1) * 100
    day5_ret = (last['Close'] / df['Close'].iloc[-5] - 1) * 100
    m1_ret = (last['Close'] / df['Close'].iloc[-20] - 1) * 100
    vol_ratio = last['Volume'] / df['Volume'].rolling(5).mean().iloc[-1]

    # C. K棒型態判定
    body_pct = (last['Close'] - last['Open']) / last['Open']
    if last['High'] == last['Low']: k_type = "一字線"
    elif body_pct > 0.03: k_type = "帶量長紅"
    elif bias_pct > 0 and last['Close'] > curr_vwma_val: k_type = "VWMA回測守穩"
    else: k_type = "一般"

    # D. 籌碼動向判定 (量價邏輯)
    if day1_ret > 0.5 and vol_ratio > 1.2: chip = "籌碼集中 (量價齊揚)"
    elif day1_ret > 0 and vol_ratio < 0.6: chip = "籌碼鎖定 (惜售) 🔒"
    elif day1_ret < -0.5 and vol_ratio > 1.2: chip = "價量背離 (虛漲) ⚠️"
    else: chip = "籌碼中性 (多頭整理)"

    # E. 複合評分系統 (0-20分)
    score = 10
    if last['Close'] > curr_vwma_val: score += 5  # 趨勢強度
    if vol_ratio > 1.2: score += 3              # 成交動能
    if day1_ret > 1.5: score += 2               # 短線爆發
    if bias_pct > 10: score = 20                # 絕對過熱 (ETF門檻較個股低)

    # F. 決策建議
    if score >= 18:
        tag = "🚀 極致飆股 (過熱)" if bias_pct > 8 else "🚀 極致飆股"
        risk = "⚠️ 極高 (過熱)" if bias_pct > 8 else "✅ 穩健 (蓄勢)"
        advice = "💰 分批停利" if bias_pct > 8 else "持股續抱"
    elif score >= 12:
        tag = "📈 強勢多頭"
        risk = "✅ 穩健"
        advice = "持股續抱"
    else:
        tag = "📉 破線轉弱"
        risk = "📉 趨勢破壞"
        advice = "❌ 觸及停損"

    return {
        "Score": score, "股性標籤": tag, "K棒": k_type,
        "風險評級": risk, "籌碼動向": chip, "量比": round(vol_ratio, 2),
        "建議": advice, "代碼": ticker, "名稱": WATCHLIST.get(ticker),
        "收盤": round(last['Close'], 2), "VWMA": round(curr_vwma_val, 2),
        "乖離%": round(bias_pct, 2), "1D%": round(day1_ret, 2),
        "5D%": round(day5_ret, 2)
    }

def run_report():
    results = []
    print(f"📊 正在生成 {datetime.now().strftime('%Y-%m-%d')} 台股 ETF 量化報告...")

    for ticker in tqdm(WATCHLIST.keys()):
        try:
            stock = yf.Ticker(ticker)
            df = stock.history(period="3mo")
            if len(df) < 20: continue
            results.append(analyze_logic(ticker, df))
        except Exception as e:
            print(f"跳過 {ticker}: {e}")

    final_df = pd.DataFrame(results).sort_values("Score", ascending=False)

    # 存檔
    file_name = f"ETF_Report_{datetime.now().strftime('%Y%m%d')}.csv"
    final_df.to_csv(file_name, index=False, encoding="utf_8_sig")

    # 終端機顯示關鍵摘要
    print("\n" + "="*100)
    display_cols = ['Score', '股性標籤', '籌碼動向', '代碼', '名稱', '收盤', '乖離%', '建議']
    print(final_df[display_cols].to_string(index=False))
    print("="*100)
    print(f"\n✅ 報告已存檔：{file_name}")

if __name__ == "__main__":
    run_report()

📊 正在生成 2026-03-26 台股 ETF 量化報告...


100%|██████████| 41/41 [00:05<00:00,  8.11it/s]


 Score   股性標籤        籌碼動向        代碼           名稱     收盤   乖離%     建議
    20 🚀 極致飆股 籌碼集中 (量價齊揚)  00901.TW     永豐智能車供應鏈  30.20  1.03   持股續抱
    17 📈 強勢多頭 籌碼鎖定 (惜售) 🔒  00728.TW      第一金工業30  45.40  4.53   持股續抱
    17 📈 強勢多頭 籌碼鎖定 (惜售) 🔒   0053.TW         元大電子 172.70  1.75   持股續抱
    17 📈 強勢多頭 籌碼鎖定 (惜售) 🔒   0056.TW        元大高股息  38.67  0.18   持股續抱
    17 📈 強勢多頭 籌碼中性 (多頭整理)  00733.TW       富邦臺灣中小  56.95  2.15   持股續抱
    17 📈 強勢多頭 籌碼鎖定 (惜售) 🔒  00891.TW      中信關鍵半導體  24.55  1.24   持股續抱
    17 📈 強勢多頭 籌碼中性 (多頭整理)  00894.TW     中信小資高價30  34.95  5.91   持股續抱
    17 📈 強勢多頭 籌碼中性 (多頭整理)  00881.TW      國泰台灣5G+  37.57  2.66   持股續抱
    17 📈 強勢多頭 籌碼中性 (多頭整理)   0051.TW      元大中型100 111.75  2.98   持股續抱
    17 📈 強勢多頭 籌碼中性 (多頭整理)  00936.TW    台新永續高息中小型  17.85  0.80   持股續抱
    17 📈 強勢多頭 籌碼中性 (多頭整理) 00981A.TW      中信成長高股息  21.19  7.69   持股續抱
    17 📈 強勢多頭 籌碼中性 (多頭整理)  00690.TW      富邦印度ETF  53.30  1.66   持股續抱
    17 📈 強勢多頭 籌碼中性 (多頭整理)  00830.TW      富邦台灣ETF  56.40  1.94   持股續抱
    17 📈 強勢多頭 籌碼鎖定 (惜售) 🔒  00905.